# Capítulo 7: Construcción de aplicaciones de chat
## Inicio rápido con la API de OpenAI

Este cuaderno está adaptado del [Repositorio de muestras de Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) que incluye cuadernos que acceden a los servicios de [Azure OpenAI](aoai-assignment.ipynb).

La API Python de OpenAI también funciona con los modelos Azure OpenAI, con algunas modificaciones. Aprende más sobre las diferencias aquí: [Cómo cambiar entre endpoints de OpenAI y Azure OpenAI con Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Visión general  
"Los grandes modelos de lenguaje son funciones que asignan texto a texto. Dada una cadena de texto de entrada, un gran modelo de lenguaje intenta predecir el texto que vendrá después"(1). Este cuaderno de "inicio rápido" introducirá a los usuarios a conceptos de alto nivel sobre LLM, requisitos básicos del paquete para comenzar con AML, una introducción suave al diseño de indicaciones, y varios ejemplos breves de diferentes casos de uso. 


## Tabla de Contenidos  

[Descripción general](#overview)  
[Cómo usar el servicio OpenAI](#how-to-use-openai-service)  
[1. Crear tu servicio OpenAI](#1.-creating-your-openai-service)  
[2. Instalación](#2.-installation)    
[3. Credenciales](#3.-credentials)  

[Casos de uso](#use-cases)    
[1. Resumir texto](#1.-summarize-text)  
[2. Clasificar texto](#2.-classify-text)  
[3. Generar nuevos nombres de productos](#3.-generate-new-product-names)  
[4. Ajustar un clasificador](#4.fine-tune-a-classifier)  

[Referencias](#references)


### Crea tu primer prompt  
Este breve ejercicio proporcionará una introducción básica para enviar prompts a un modelo de OpenAI para una tarea simple de "resumen".


**Pasos**:  
1. Instala la biblioteca OpenAI en tu entorno de Python  
2. Carga las bibliotecas auxiliares estándar y establece tus credenciales de seguridad típicas de OpenAI para el Servicio de OpenAI que has creado  
3. Elige un modelo para tu tarea  
4. Crea un prompt simple para el modelo  
5. ¡Envía tu solicitud a la API del modelo!


### 1. Instalar OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Importar bibliotecas de ayuda e instanciar credenciales


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. Encontrar el modelo adecuado  
Modelos como GPT-4o y GPT-4o mini pueden entender y generar lenguaje natural, y están disponibles en la plataforma de OpenAI con diferentes niveles de potencia y velocidad adecuados para distintas tareas.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Diseño de prompts  

"La magia de los grandes modelos de lenguaje es que, al ser entrenados para minimizar este error de predicción sobre grandes cantidades de texto, los modelos terminan aprendiendo conceptos útiles para estas predicciones. Por ejemplo, aprenden conceptos como"(1):

* cómo deletrear
* cómo funciona la gramática
* cómo parafrasear
* cómo responder preguntas
* cómo mantener una conversación
* cómo escribir en muchos idiomas
* cómo programar
* etc.

#### Cómo controlar un gran modelo de lenguaje  
"De todas las entradas a un gran modelo de lenguaje, el más influyente con diferencia es el prompt de texto(1).

Se puede pedir a los grandes modelos de lenguaje que produzcan salida de varias maneras:

Instrucción: Dígale al modelo lo que quiere
Completado: Induzca al modelo a completar el comienzo de lo que quiere
Demostración: Muestre al modelo lo que quiere, con:
Unos pocos ejemplos en el prompt
Cientos o miles de ejemplos en un conjunto de datos de entrenamiento de ajuste fino"



#### Hay tres pautas básicas para crear prompts:

**Mostrar y decir**. Aclare lo que quiere, ya sea a través de instrucciones, ejemplos o una combinación de ambos. Si quiere que el modelo clasifique una lista de elementos en orden alfabético o que clasifique un párrafo según el sentimiento, demuéstrele que eso es lo que quiere.

**Proporcione datos de calidad**. Si intenta construir un clasificador o hacer que el modelo siga un patrón, asegúrese de que haya suficientes ejemplos. Asegúrese de revisar sus ejemplos — el modelo suele ser lo suficientemente inteligente para detectar errores ortográficos básicos y darle una respuesta, pero también podría asumir que esto es intencional y puede afectar la respuesta.

**Verifique sus ajustes.** Los ajustes de temperatura y top_p controlan qué tan determinista es el modelo al generar una respuesta. Si le está pidiendo una respuesta donde solo hay una respuesta correcta, entonces querrá establecer estos valores más bajos. Si busca respuestas más diversas, entonces podría querer establecerlos más altos. El error número uno que la gente comete con estos ajustes es asumir que son controles de "inteligencia" o "creatividad".


Fuente: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. ¡Enviar!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Repetir la misma llamada, ¿cómo se comparan los resultados?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Resumir texto  
#### Desafío  
Resume texto añadiendo un 'tl;dr:' al final de un pasaje de texto. Observa cómo el modelo entiende cómo realizar varias tareas sin instrucciones adicionales. Puedes experimentar con indicaciones más descriptivas que tl;dr para modificar el comportamiento del modelo y personalizar el resumen que recibes(3).  

Trabajos recientes han demostrado ganancias sustanciales en muchas tareas y puntos de referencia de PLN mediante un preentrenamiento en un gran corpus de texto seguido de un ajuste fino para una tarea específica. Aunque típicamente es independiente de la tarea en arquitectura, este método aún requiere conjuntos de datos de ajuste fino específicos de la tarea con miles o decenas de miles de ejemplos. En contraste, los humanos generalmente pueden realizar una nueva tarea lingüística con sólo unos pocos ejemplos o instrucciones simples, algo que los sistemas actuales de PLN aún luchan en gran medida por hacer. Aquí mostramos que aumentar la escala de los modelos de lenguaje mejora considerablemente el rendimiento independiente de la tarea con pocos ejemplos, a veces incluso alcanzando competitividad con métodos anteriores de ajuste fino que eran el estado del arte.  



Tl;dr  


# Ejercicios para varios casos de uso  
1. Resumir texto  
2. Clasificar texto  
3. Generar nuevos nombres de productos


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Clasificar Texto  
#### Desafío  
Clasificar elementos en categorías proporcionadas en el momento de la inferencia. En el siguiente ejemplo, proporcionamos tanto las categorías como el texto a clasificar en el aviso (*playground_reference). 

Consulta del cliente: Hola, una de las teclas de mi teclado de laptop se rompió recientemente y necesitaré un reemplazo:

Categoría clasificada:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Generar Nuevos Nombres de Producto
#### Desafío
Crear nombres de productos a partir de palabras de ejemplo. Aquí incluimos en la solicitud información sobre el producto para el que vamos a generar nombres. También proporcionamos un ejemplo similar para mostrar el patrón que deseamos recibir. Además, hemos establecido un valor de temperatura alto para aumentar la aleatoriedad y obtener respuestas más innovadoras.

Descripción del producto: Una máquina casera para hacer batidos
Palabras clave: rápido, saludable, compacto.
Nombres de producto: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Descripción del producto: Un par de zapatos que pueden adaptarse a cualquier tamaño de pie.
Palabras clave: adaptable, apto, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Referencias  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Portal Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Mejores prácticas para ajustar modelos GPT para clasificar texto](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Para Más Ayuda  
[Equipo de Comercialización de OpenAI](AzureOpenAITeam@microsoft.com) 


# Colaboradores
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
